<a href="https://colab.research.google.com/github/yogeshwardev/CSA6102-Digital_forensics-/blob/main/lab%20programs%20outputs%20from%20colab%201%20to%2013.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, shutil, time

# Create an "original" piece of evidence
with open("evidence_original.txt", "w") as f:
    f.write("Case File #001 - Suspect device log")

orig_stat = os.stat("evidence_original.txt")
print("Original file - created (ctime):", time.ctime(orig_stat.st_ctime))
print("Original file - modified (mtime):", time.ctime(orig_stat.st_mtime))

# Simulate an investigator copying the file to a workstation
time.sleep(1)
shutil.copy2("evidence_original.txt", "evidence_copy.txt")

copy_stat = os.stat("evidence_copy.txt")
print("\nCopied file - created (ctime):", time.ctime(copy_stat.st_ctime))
print("Copied file - modified (mtime):", time.ctime(copy_stat.st_mtime))

print("\nLocard's Exchange Principle: the copy operation itself created a new")
print("ctime on the destination file - every interaction leaves a trace.")


Original file - created (ctime): Thu Jul 16 05:24:48 2026
Original file - modified (mtime): Thu Jul 16 05:24:48 2026

Copied file - created (ctime): Thu Jul 16 05:24:49 2026
Copied file - modified (mtime): Thu Jul 16 05:24:48 2026

Locard's Exchange Principle: the copy operation itself created a new
ctime on the destination file - every interaction leaves a trace.


In [ ]:
import hashlib, datetime

def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Step 1: Investigator seizes evidence and hashes it immediately
with open("evidence.txt", "w") as f:
    f.write("Suspect chat log: meeting at 10pm, bring the drive.")

seizure_hash = sha256_of("evidence.txt")
custody_log = []
custody_log.append(f"{datetime.datetime.now()} - SEIZED by Officer A - hash={seizure_hash}")

# Step 2: Evidence is later handed to a forensic analyst; verify integrity
handoff_hash = sha256_of("evidence.txt")

if handoff_hash == seizure_hash:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity VERIFIED")
else:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity FAILED (tampered!)")

print("=== Chain of Custody Log ===")
for entry in custody_log:
    print(entry)


=== Chain of Custody Log ===
2026-07-16 05:24:49.498436 - SEIZED by Officer A - hash=b7eb05cf41efbcc50adc36b12679ce4f12de58460e59500eb6bb62858810f84c
2026-07-16 05:24:49.498663 - RECEIVED by Analyst B - integrity VERIFIED


In [ ]:
import re

def check_email(sender, subject, body):
    flags = []
    if re.search(r"(urgent|verify your account|suspended|click here)", body, re.IGNORECASE):
        flags.append("Urgency/pressure language detected")

    if re.search(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", body):
        flags.append("Raw IP address link found in body")

    domain = sender.split("@")[-1]
    if any(k in domain.lower() for k in ["secure","verify","update"]) and "@gmail" not in sender:
        flags.append("Suspicious sender domain naming pattern")

    return flags

emails = [
    ("support@paypal.com", "Your monthly statement", "Please find your statement attached."),
    ("alert@paypal-secure-verify.com", "URGENT: Verify your account", "Click here: http://192.168.10.5/login"),
]

for sender, subject, body in emails:
    issues = check_email(sender, subject, body)
    verdict = "PHISHING SUSPECTED" if issues else "Looks legitimate"
    print(f"\nFrom: {sender}\nSubject: {subject}\nVerdict: {verdict}")
    for i in issues:
        print(" -", i)



From: support@paypal.com
Subject: Your monthly statement
Verdict: Looks legitimate

From: alert@paypal-secure-verify.com
Subject: URGENT: Verify your account
Verdict: PHISHING SUSPECTED
 - Urgency/pressure language detected
 - Raw IP address link found in body
 - Suspicious sender domain naming pattern


In [ ]:
import socket

domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]
print("OSINT Domain Reconnaissance")

for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")


OSINT Domain Reconnaissance
www.google.com                 -> 142.251.154.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


In [ ]:
import datetime

def investigation_report(case_id, evidence_list):
    stages = {
        "Identification": f"Incident reported for case {case_id}. Devices/logs identified for review.",
        "Collection": f"{len(evidence_list)} item(s) collected: {', '.join(evidence_list)}",
        "Preservation": "All items hashed (SHA-256) and stored in a write-protected evidence folder.",
        "Analysis": "Log files and file metadata examined for indicators of compromise.",
        "Reporting": "Findings compiled into a structured forensic report for legal review.",
    }

    print(f"=== Investigation Lifecycle: Case {case_id} ===")
    print(f"Generated: {datetime.datetime.now()}\n")

    for stage, detail in stages.items():
        print(f"[{stage}]")
        print(f" {detail}\n")

investigation_report("CASE-2026-014", ["laptop_disk_image.dd", "router_traffic.pcap", "email_headers.txt"])


=== Investigation Lifecycle: Case CASE-2026-014 ===
Generated: 2026-07-16 05:24:49.564947

[Identification]
 Incident reported for case CASE-2026-014. Devices/logs identified for review.

[Collection]
 3 item(s) collected: laptop_disk_image.dd, router_traffic.pcap, email_headers.txt

[Preservation]
 All items hashed (SHA-256) and stored in a write-protected evidence folder.

[Analysis]
 Log files and file metadata examined for indicators of compromise.

[Reporting]
 Findings compiled into a structured forensic report for legal review.



In [ ]:
import hashlib

def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Create a sample "disk" file (simulating a small storage device)
with open("original_disk.img", "wb") as f:
    f.write(b"HEADER" + bytes(range(256)) * 4 + b"FOOTER")

# Step 1: Create a bit-for-bit forensic copy (bit-stream image)
with open("original_disk.img", "rb") as src, open("forensic_copy.img", "wb") as dst:
    dst.write(src.read())

# Step 2: Verify the copy is identical using hashing
original_hash = sha256_of("original_disk.img")
copy_hash = sha256_of("forensic_copy.img")

print("Original image hash:", original_hash)
print("Forensic copy hash :", copy_hash)
print("Match:", "VERIFIED - exact bit-stream copy" if original_hash == copy_hash else "MISMATCH - copy corrupted")


Original image hash: d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Forensic copy hash : d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Match: VERIFIED - exact bit-stream copy


In [ ]:
SIGNATURES = {
    b"\xFF\xD8\xFF": "JPEG image",
    b"\x89PNG": "PNG image",
    b"%PDF": "PDF document",
    b"PK\x03\x04": "ZIP archive (or .docx/.xlsx)",
}

def identify_file(path):
    with open(path, "rb") as f:
        header = f.read(8)
        for sig, filetype in SIGNATURES.items():
            if header.startswith(sig):
                return filetype
        return "Unknown file type"

# Create sample files with fake/renamed extensions to test signature detection
with open("photo.txt", "wb") as f: # renamed JPEG
    f.write(b"\xFF\xD8\xFF\xE0" + b"\x00" * 20)

with open("document.dat", "wb") as f: # renamed PDF
    f.write(b"%PDF-1.4" + b"\x00" * 20)

for filename in ["photo.txt", "document.dat"]:
    print(f"{filename:15s} -> Actual type: {identify_file(filename)}")


photo.txt       -> Actual type: JPEG image
document.dat    -> Actual type: PDF document


In [ ]:
import os

# Step 1: Create a file, then "delete" it (simulating accidental/malicious deletion)
with open("secret_note.txt", "w") as f:
    f.write("MEETING_POINT: warehouse 7, 11pm")

with open("secret_note.txt", "rb") as f:
    original_bytes = f.read()

os.remove("secret_note.txt")

print("File 'secret_note.txt' deleted.")
print("Exists on disk now?", os.path.exists("secret_note.txt"))

# Step 2: Simulate raw disk scanning - in real forensics, tools like Autopsy
# scan unallocated space for byte patterns. Here we search a "disk buffer"
# (which still holds the bytes) for the recoverable content.
disk_buffer = original_bytes + b"\x00" * 50 # unallocated space padding
marker = b"MEETING_POINT"
index = disk_buffer.find(marker)

if index != -1:
    recovered = disk_buffer[index:].split(b"\x00")[0]
    with open("recovered_note.txt", "wb") as f:
        f.write(recovered)
    print("Recovered content:", recovered.decode())
else:
    print("No recoverable data found.")


File 'secret_note.txt' deleted.
Exists on disk now? False
Recovered content: MEETING_POINT: warehouse 7, 11pm


In [ ]:
import re
from collections import Counter

# Create the sample system log so the script runs standalone
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")

failed_by_ip = Counter()
with open("login_attempts.log") as f:
    for line in f:
        if "LOGIN FAILED" in line:
            match = re.search(r"ip=(\S+)", line)
            if match:
                failed_by_ip[match.group(1)] += 1

print("System log forensic analysis - failed logins by source IP:")
for ip, count in failed_by_ip.items():
    print(f" {ip}: {count} failed attempts")
    if count >= 5:
        print(f" -> Evidence of brute-force intrusion attempt from {ip}")


System log forensic analysis - failed logins by source IP:
 203.0.113.99: 5 failed attempts
 -> Evidence of brute-force intrusion attempt from 203.0.113.99


In [ ]:
raw_header = """Delivered-To: victim@example.com
Received: from mail-relay-99.suspicious-host.ru (unknown [45.33.32.156])
by mx.example.com; Tue, 08 Jul 2026 09:15:00 +0000
From: \"Bank Support\" <support@paypal-secure-verify.com>
Reply-To: attacker@totallynotscam.net
Subject: Urgent: Verify your account
"""

def analyze_header(header_text):
    findings = []
    for line in header_text.splitlines():
        if line.startswith("Received:") and "suspicious" in line.lower():
            findings.append(f"Suspicious relay server found: {line.strip()}")
        if line.startswith("Reply-To:"):
            findings.append(f"Reply-To differs from sender - possible spoofing: {line.strip()}")
        if line.startswith("From:") and "-secure-verify" in line:
            findings.append(f"Sender domain uses suspicious naming: {line.strip()}")
    return findings

print("Email Header Forensic Analysis")
for finding in analyze_header(raw_header):
    print(" -", finding)


Email Header Forensic Analysis
 - Suspicious relay server found: Received: from mail-relay-99.suspicious-host.ru (unknown [45.33.32.156])
 - Sender domain uses suspicious naming: From: "Bank Support" <support@paypal-secure-verify.com>
 - Reply-To differs from sender - possible spoofing: Reply-To: attacker@totallynotscam.net


In [ ]:
# Create a sample simulated packet log so this runs standalone
with open("traffic.log", "w") as f:
    f.write("10.0.0.9 -> 10.0.0.5:22 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:23 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:25 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:443 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 ACK\n")

# Count distinct destination ports probed by each source IP
ports_by_source = {}
with open("traffic.log") as f:
    for line in f:
        parts = line.split()
        src = parts[0]
        dst_port = parts[2].split(":")[1]
        ports_by_source.setdefault(src, set()).add(dst_port)

print("Network traffic forensic analysis:")
for src, ports in ports_by_source.items():
    print(f" {src}: contacted {len(ports)} distinct port(s) -> {sorted(ports)}")
    if len(ports) >= 4:
        print(f" -> ALERT: {src} shows port-scanning behaviour")


Network traffic forensic analysis:
 10.0.0.9: contacted 5 distinct port(s) -> ['22', '23', '25', '443', '80']
 -> ALERT: 10.0.0.9 shows port-scanning behaviour
 10.0.0.20: contacted 1 distinct port(s) -> ['80']


In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS
# Install once: pip install Pillow

def create_sample_image(path):
    # Creates a small test image (a real photo would already have EXIF data)
    img = Image.new("RGB", (100, 100), color="blue")
    img.save(path)

def show_metadata(path):
    img = Image.open(path)
    print("File:", path)
    print("Format:", img.format)
    print("Size:", img.size)
    print("Mode:", img.mode)

    exif_data = img._getexif() if hasattr(img, "_getexif") else None

    if exif_data:
        for tag_id, value in exif_data.items():
            tag = TAGS.get(tag_id, tag_id)
            print(f" {tag}: {value}")
    else:
        print(" No EXIF metadata found (this sample image has none).")
        print(" Real photos from phones/cameras typically contain GPS,")
        print(" device model, and timestamp data - valuable forensic evidence.")

create_sample_image("sample.jpg")
show_metadata("sample.jpg")


File: sample.jpg
Format: JPEG
Size: (100, 100)
Mode: RGB
 No EXIF metadata found (this sample image has none).
 Real photos from phones/cameras typically contain GPS,
 device model, and timestamp data - valuable forensic evidence.


In [ ]:
import datetime

def generate_forensic_report(case_id, findings, investigator):
    lines = []
    lines.append("DIGITAL FORENSIC INVESTIGATION REPORT")
    lines.append(f"Case ID: {case_id}")
    lines.append(f"Investigator: {investigator}")
    lines.append(f"Report generated: {datetime.datetime.now()}")
    lines.append("-" * 45)
    lines.append("FINDINGS:")
    for i, finding in enumerate(findings, 1):
        lines.append(f" {i}. {finding}")
    lines.append("-" * 45)
    lines.append("CONCLUSION:")
    lines.append(" Evidence indicates unauthorized access consistent with a")
    lines.append(" brute-force attack. Recommend IP block and password reset.")
    lines.append(" This report is suitable for legal review and expert testimony.")
    return "\n".join(lines)

findings = [
    "203.0.113.99 attempted 5 failed logins within 20 seconds (log file evidence).",
    "SHA-256 hash of evidence file verified unchanged throughout custody.",
    "Email header traced to a spoofed domain via suspicious relay server.",
]

report = generate_forensic_report("CASE-2026-014", findings, "Analyst B")
print(report)

with open("forensic_report.txt", "w") as f:
    f.write(report)


DIGITAL FORENSIC INVESTIGATION REPORT
Case ID: CASE-2026-014
Investigator: Analyst B
Report generated: 2026-07-16 05:24:49.701411
---------------------------------------------
FINDINGS:
 1. 203.0.113.99 attempted 5 failed logins within 20 seconds (log file evidence).
 2. SHA-256 hash of evidence file verified unchanged throughout custody.
 3. Email header traced to a spoofed domain via suspicious relay server.
---------------------------------------------
CONCLUSION:
 Evidence indicates unauthorized access consistent with a
 brute-force attack. Recommend IP block and password reset.
 This report is suitable for legal review and expert testimony.
